In [2]:
print("hello world")

hello world


In [4]:
import pandas as pd

# Load your local CSV file
df = pd.read_csv('used_cars.csv')

# Show the first few rows to confirm it loaded correctly
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [6]:
print(df.columns.tolist())

['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']


In [7]:
# 1. Count the occurrences using the correct lowercase column name
model_counts = df['model'].value_counts()

# 2. Filter for car models that appear less than 5 times
rare_models = model_counts[model_counts < 5]

# 3. Print the results
print(f"There are {len(rare_models)} car models with less than 5 entries:\n")
print(rare_models)

There are 1737 car models with less than 5 entries:

model
GV70 3.5T Sport               4
340 i                         4
AMG C 63 S                    4
300 Touring                   4
A5 2.0T Premium Plus          4
                             ..
X3 xDrive28i                  1
CT 200h Base                  1
Martin DB7 Vantage Volante    1
Impala 2LZ                    1
Taycan                        1
Name: count, Length: 1737, dtype: int64


In [8]:
# Filter rows where the model contains 'supra' OR 'innova'
filtered_cars = df[df['model'].str.contains('supra|innova', case=False, na=False)]

# Display how many entries were found
print(f"Found {len(filtered_cars)} entries for Supra/Innova:")

# Show the matching rows
filtered_cars

Found 4 entries for Supra/Innova:


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
12,Toyota,Supra 3.0 Premium,2021,"12,500 mi.",Gasoline,382.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,A/T,Yellow,Black,None reported,Yes,"$53,500"
706,Toyota,Supra A91-MT Edition,2023,"1,500 mi.",Gasoline,382.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,6-Speed M/T,Gray,Orange,None reported,Yes,"$84,900"
787,Toyota,Supra A91 Edition,2021,"6,400 mi.",Gasoline,382.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,Transmission w/Dual Shift Mode,Gray,Black,None reported,Yes,"$68,000"
1215,Toyota,Supra 3.0 Premium,2022,"24,850 mi.",Gasoline,382.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,A/T,Black,Black,None reported,Yes,"$53,568"


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# ==========================================
# 1. LOAD AND CLEAN THE DATASET
# ==========================================
print("Loading and cleaning dataset...")
df = pd.read_csv('used_cars.csv')

# Drop initial missing rows in crucial columns to avoid errors
df = df.dropna(subset=['price', 'model', 'brand', 'milage'])

# --- REPAIR MILEAGE COLUMN (Convert '73,049 mi.' to 73049.0) ---
df['milage'] = df['milage'].astype(str).str.replace(',', '', regex=False)
df['milage'] = df['milage'].str.replace('mi.', '', case=False, regex=False)
df['milage'] = df['milage'].str.strip()
df['milage'] = pd.to_numeric(df['milage'], errors='coerce')

# --- REPAIR PRICE COLUMN (Convert '$30,510' to 30510.0) ---
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False)
df['price'] = df['price'].str.replace(',', '', regex=False)
df['price'] = df['price'].str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Drop rows where mileage or price couldn't be parsed into numbers after cleaning
df = df.dropna(subset=['milage', 'price'])

# ==========================================
# 2. MERGE RARE MODELS INTO 'OTHER'
# ==========================================
print("Grouping rare car models...")
# Count the occurrences of each model
model_counts = df['model'].value_counts()

# Identify models with less than 10 entries (like the Supra)
rare_models = model_counts[model_counts < 10].index

# Replace rare model names with 'other'
df['model'] = df['model'].replace(rare_models, 'other')

# ==========================================
# 3. SELECT FEATURES AND TARGET
# ==========================================
categorical_cols = ['brand', 'model', 'fuel_type', 'transmission']
numerical_cols = ['model_year', 'milage'] 

X = df[categorical_cols + numerical_cols]
y = df['price']

# ==========================================
# 4. ONE-HOT ENCODING (Convert Text to Numbers)
# ==========================================
print("Performing One-Hot Encoding...")
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# ==========================================
# 5. TRAIN-TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# ==========================================
# 6. FEATURE SCALING
# ==========================================
print("Scaling numerical features...")
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

# ==========================================
# 7. TRAIN LINEAR REGRESSION & EVALUATE
# ==========================================
print("Training Linear Regression model...")
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Generate predictions using the test data split
y_pred = model.predict(X_test_scaled)

# Calculate final metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# ==========================================
# 8. VIEW FINAL RESULTS
# ==========================================
print("\n" + "="*40)
print("FINAL MODEL PERFORMANCE")
print("="*40)
print(f"R² Score (Accuracy): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print("="*40)

Loading and cleaning dataset...
Grouping rare car models...
Performing One-Hot Encoding...
Scaling numerical features...
Training Linear Regression model...

FINAL MODEL PERFORMANCE
R² Score (Accuracy): 0.0699
Mean Absolute Error (MAE): 24086.70


In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor  # <-- Swapped!
from sklearn.metrics import r2_score, mean_absolute_error

# ==========================================
# 1. LOAD AND CLEAN THE DATASET
# ==========================================
print("Loading and cleaning dataset...")
df = pd.read_csv('used_cars.csv')

df = df.dropna(subset=['price', 'model', 'brand', 'milage'])

# Clean Mileage
df['milage'] = df['milage'].astype(str).str.replace(',', '', regex=False)
df['milage'] = df['milage'].str.replace('mi.', '', case=False, regex=False)
df['milage'] = df['milage'].str.strip()
df['milage'] = pd.to_numeric(df['milage'], errors='coerce')

# Clean Price
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False)
df['price'] = df['price'].str.replace(',', '', regex=False)
df['price'] = df['price'].str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')

df = df.dropna(subset=['milage', 'price'])

# --- NEW: FILTER OUT EXTREME OUTLIERS TO FIX TRAINING ---
# Keep only cars priced between $1,000 and $150,000 (adjust based on currency)
df = df[(df['price'] > 1000) & (df['price'] < 150000)]
# --------------------------------------------------------

# ==========================================
# 2. MERGE RARE MODELS INTO 'OTHER'
# ==========================================
print("Grouping rare car models...")
model_counts = df['model'].value_counts()
rare_models = model_counts[model_counts < 10].index
df['model'] = df['model'].replace(rare_models, 'other')

# ==========================================
# 3. SELECT FEATURES AND TARGET
# ==========================================
categorical_cols = ['brand', 'model', 'fuel_type', 'transmission']
numerical_cols = ['model_year', 'milage'] 

X = df[categorical_cols + numerical_cols]
y = df['price']

# ==========================================
# 4. ONE-HOT ENCODING
# ==========================================
print("Performing One-Hot Encoding...")
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# ==========================================
# 5. TRAIN-TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# ==========================================
# 6. TRAIN RANDOM FOREST MODEL & EVALUATE
# ==========================================
print("Training Random Forest Regressor model...")
# Random Forest handles columns natively and doesn't strictly require scaling
model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Generate predictions
y_pred = model.predict(X_test)

# Calculate final metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# ==========================================
# 7. VIEW FINAL RESULTS
# ==========================================
print("\n" + "="*40)
print("UPDATED MODEL PERFORMANCE (RANDOM FOREST)")
print("="*40)
print(f"R² Score (Accuracy): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print("="*40)

Loading and cleaning dataset...
Grouping rare car models...
Performing One-Hot Encoding...
Training Random Forest Regressor model...

UPDATED MODEL PERFORMANCE (RANDOM FOREST)
R² Score (Accuracy): 0.5901
Mean Absolute Error (MAE): 11112.81


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# ==========================================
# 1. LOAD AND CLEAN DATA
# ==========================================
print("Loading dataset...")
df = pd.read_csv('used_cars.csv')

# Drop basic missing pieces
df = df.dropna(subset=['price', 'model', 'brand', 'milage', 'engine', 'model_year'])

# Clean Mileage & Price strings
df['milage'] = pd.to_numeric(df['milage'].astype(str).str.replace(',', '').str.replace('mi.', '', case=False).str.strip(), errors='coerce')
df['price'] = pd.to_numeric(df['price'].astype(str).str.replace('$', '').str.replace(',', '').str.strip(), errors='coerce')

# Filter out extreme outliers to stabilize training
df = df[(df['price'] > 1000) & (df['price'] < 150000)]

# ==========================================
# 2. FEATURE ENGINEERING (New Boosters!)
# ==========================================
print("Engineering new features...")

# Feature 1: Extract Engine Size (e.g., '3.5T' or '3.0L' -> 3.5 or 3.0)
df['engine_size'] = df['engine'].astype(str).str.extract(r'(\d+\.\d+)').astype(float)
# If a car doesn't have a decimal engine size listed, fill it with the median
df['engine_size'] = df['engine_size'].fillna(df['engine_size'].median())

# Feature 2: Calculate Car Age instead of raw year
df['car_age'] = 2026 - df['model_year']

# Drop records that became NaN during cleaning
df = df.dropna(subset=['milage', 'price', 'engine_size'])

# ==========================================
# 3. MERGE RARE MODELS
# ==========================================
model_counts = df['model'].value_counts()
rare_models = model_counts[model_counts < 10].index
df['model'] = df['model'].replace(rare_models, 'other')

# ==========================================
# 4. SELECT FILTERED COLUMNS
# ==========================================
# Notice we dropped raw model_year and added car_age + engine_size
categorical_cols = ['brand', 'model', 'fuel_type', 'transmission']
numerical_cols = ['car_age', 'milage', 'engine_size'] 

X = df[categorical_cols + numerical_cols]
y = df['price']

# ==========================================
# 5. ENCODING & SPLIT
# ==========================================
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# ==========================================
# 6. TRAIN HYPER-TUNED RANDOM FOREST
# ==========================================
print("Training Optimized Random Forest...")
# Tuned max_features to reduce overfitting on the encoded 'other' categories
model = RandomForestRegressor(
    n_estimators=150, 
    max_depth=20, 
    max_features='sqrt', 
    random_state=42, 
    n_jobs=-1
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*40)
print("OPTIMIZED PERFORMANCE RESULTS")
print("="*40)
print(f"R² Score (Accuracy): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print("="*40)

Loading dataset...
Engineering new features...
Training Optimized Random Forest...

OPTIMIZED PERFORMANCE RESULTS
R² Score (Accuracy): 0.7707
Mean Absolute Error (MAE): 8378.48
